In [ ]:
import sys
sys.path.append('../../../src/')

import matplotlib.pyplot as plt

# Import methods
import QGP.Defocusing.Methods as Def
import QGP.Focusing.Methods as Foc
from QGP.Soliton_Gen import *
from Utilities import *

import config

# Bright

In [ ]:
L = 80
N = 10_001
Nt = 10_000 #10_000
tf = 15

k = 1/4

pars = {"k":k}
c = np.sqrt(2)/2
w = 1 #np.sqrt(2)/2

w_bright_theo, x_bright = soliton(c, k, w=w, soliton_type="bright", L=L, N=N, as_func=True)

In [ ]:
def bright_soliton(x, t, c=c, w=w, x_0=0):
    i = 1j
    output = w_moving_theo(x-c*t-x_0) * np.exp(i*(-c**2/4*t-w*t))
    return output

In [ ]:
w_bright = Foc.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_bright.set_w_theo(bright_soliton)
w_bright.solve()

In [ ]:
def sol_gauss(x, A):
    output = w_bright_theo(x) - A*np.exp(-x**2)
    return output

A = 0.02041595129863531 #7e-2#roots[1]-0.02
init_cond = lambda x: sol_gauss(x, A)

w_bright_gauss_007 = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_bright_gauss_007.set_init_condition(init_cond)
w_bright_gauss_007.solve()

In [ ]:
w_bright.w_theo = bright_soliton
w_bright_gauss_007.w_theo = bright_soliton

L2_errors_bright = w_bright.get_error()
L2_errors_bright_gauss_007 = w_bright_gauss_007.get_error()

In [ ]:
ms = .5
plt.plot(w_bright.t, L2_errors_bright, ".", label="Simulation", ms=ms)
plt.plot(w_bright.t, L2_errors_bright_gauss_007, ".", label=r"$A={}$".format(A), ms=ms)

plt.xlabel(r"$t$")
plt.ylabel(r"$L^2$-diff", rotation='horizontal', ha='right')
#plt.yscale("log")
plt.legend()
plt.title(r"Bright soliton $c=\sqrt{2}/2, \kappa=1/4, L=80, \omega=1$")
plt.savefig("diffs/bright.png", dpi=300, bbox_inches = "tight")
plt.show()

# Dark in $D_1$

In [ ]:
k = 1/4
pars = {"k":k}
c = np.sqrt(2)/2

In [ ]:
L = 80
N = 10_001
Nt = 10_000 #10_000
tf = 15

w_dark_st_D1, x_dark_st_D1 = soliton(c, k, soliton_type="dark", L=L, N=N, as_func=True)

w_st_D1 = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_st_D1.set_init_condition(w_dark_st_D1)
w_st_D1.solve()

In [ ]:
import scipy.integrate as integrate

i = 1j
def g(x, A):
    return A * np.exp(-x**2)

def g_1(x, A):
    return -2 * A * x * np.exp(-x**2)

def g_2(x, A):
    return -2 * A * np.exp(-x**2) + 4 * A * x**2 * np.exp(-x**2)

def inner(x, y):
    return np.real(np.conjugate(x) * y)

def integrand(x, A, k, c=c):
    h = x[1] - x[0]
    u = w_dark_st_D1(x)
    u_1 = np.gradient(u, h)
    u_2 = np.gradient(u_prime, h)
    
    g_ = g(x, A)
    g_1_ = g_1(x, A)
    g_2_ = g_2(x, A)

    output_1 = inner(u, i*(g_*(1-norm(u + g_)) +g_2_) - c*g_1_)
    output_2 = k * inner(u_2*g_+2*u_1*g_1_+u*g_2_, i*(norm(u) + g_**2 + 2*inner(u,g_)))
    return output_1 - output_2

def m_t_0(x, A, k):
    J = x.size
    #h = x[1] - x[0]
    #integrand_ = integrand(x, A, k)

    output = integrate.trapezoid(integrand(x, A, k), x)
    #integ = h/2 * (integrand_[0] + integrand_[1])
    
    #for j in range(J-1):
    #    integ = integ + (h/2 * (integrand_[j] + integrand_[j+1]))

    return 2*output#integ

In [ ]:
slope = m_t_0(x_dark_st_D1, A, k)
m_0 = A * (np.pi/2)**(1/4)
slope

In [ ]:
w_st_D1.plot(w_st_D1.t[-1])

In [ ]:
def theo_dark_st_D1(x, t):
    output = w_dark_st_D1(x - c*t)
    return output

In [ ]:
j = 5000
plt.plot(w_st_D1.x, norm(w_st_D1.w[j]), marker="none", ls="-")
plt.plot(w_st_D1.x, norm(theo_dark_st_D1(w_st_D1.x, w_st_D1.t[j])), marker="none", ls="-", label="theo")
plt.legend()
plt.xlim(-10, 10)

plt.show()

In [ ]:
def sol_gauss(x, A):
    output = w_dark_st_D1(x) + A*np.exp(-x**2)
    return output

A = 2e-2 #0.02041595129863531 #7e-2#roots[1]-0.02
init_cond = lambda x: sol_gauss(x, A)

w_st_D1_gauss_007 = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_st_D1_gauss_007.set_init_condition(init_cond)
w_st_D1_gauss_007.solve()#method="fixed point")

In [ ]:
w_st_D1.w_theo = theo_dark_st_D1
w_st_D1_gauss_007.w_theo = theo_dark_st_D1
#w_in_gauss_001.w_theo = theo_dark_in
#w_in_gauss_0001.w_theo = theo_dark_in
#w_in_gauss_00001.w_theo = theo_dark_in

L2_errors_st_D1 = w_st_D1.get_error()#np.zeros(len(w_in.w)-1)
L2_errors_st_D1_gauss_007 = w_st_D1_gauss_007.get_error()#np.zeros(len(w_in.w)-1)
#errors_gauss_001 = w_in_gauss_001.get_error()#np.zeros(len(w_in.w)-1)
#errors_gauss_0001 = w_in_gauss_0001.get_error()#np.zeros(len(w_in.w)-1)
#errors_gauss_00001 = w_in_gauss_00001.get_error()#np.zeros(len(w_in.w)-1)


In [ ]:
slope

In [ ]:
slope = m_t_0(x_dark_st_D1, A, k)
m_0 = A * (np.pi/2)**(1/4)

ms = .5

plt.plot(0, m_0**2, ms=2)
plt.axline((0, m_0**2), slope=slope, linewidth=1)
plt.plot(w_st_D1.t, L2_errors_st_D1**2, ".", label="Standard", ms=ms)
plt.plot(w_st_D1.t, L2_errors_st_D1_gauss_007**2, ".", label=r"$A={}$".format(A), ms=ms)
#plt.plot(w_in.t, errors_gauss_001, ".", label=r"Gauss $A=10^{-2}$", ms=ms)
#plt.plot(w_in.t, errors_gauss_0001, ".", label=r"Gauss $A=10^{-3}$", ms=ms)
#plt.plot(w_in.t, errors_gauss_00001, ".", label=r"Gauss $A=10^{-4}$", ms=ms)
plt.xlabel(r"$t$")
plt.ylabel(r"$L^2$-diff$^2$", rotation='horizontal', ha='right')
plt.title(r"Dark soliton in $D_1, c=\sqrt{2}/2, \kappa=1/4, L=80$")
#plt.yscale("log")
plt.legend()
plt.savefig("diffs/Dark_D1.png", dpi=300, bbox_inches = "tight")
plt.show()

# Dark in $D_2$

In [ ]:
k = -50
pars = {"k":k}
c_0 = np.sqrt(2)/2

P_p_ = lambda c: P_p(c, k)
P_pp_ = lambda c: P_pp(c, k)

c_star, error = newton_1D(c_0, P_p_, P_pp_)
print("c* = ", c_star)

eps = 0.1
c_st = c_star + eps

In [ ]:
c_st

In [ ]:
L = 400 #80
N = 10_001
Nt = 10_000 #10_000
tf = 2#1 #15

w_dark_st, x_dark_st = soliton(c_st, k, soliton_type="dark", L=L, N=N, as_func=True)

In [ ]:
i = 1j
def g(x, A):
    return A * np.exp(-x**2)

def g_1(x, A):
    return -2 * A * x * np.exp(-x**2)

def g_2(x, A):
    return -2 * A * np.exp(-x**2) + 4 * A * x**2 * np.exp(-x**2)

def inner(x, y):
    return np.real(np.conjugate(x) * y)

def integrand(x, A, k, c=c_st):
    h = x[1] - x[0]
    u = w_dark_st(x)
    u_1 = np.gradient(u, h)
    u_2 = np.gradient(u_prime, h)
    
    g_ = g(x, A)
    g_1_ = g_1(x, A)
    g_2_ = g_2(x, A)

    output_1 = inner(u, i*(g_*(1-norm(u + g_)) -g_2_) - c*g_1_)
    output_2 = k * inner(u_2*g_+2*u_1*g_1_+u*g_2_, i*(norm(u) + g_**2 + u*g_))
    return output_1 + output_2

def m_t_0(x, A, k):
    J = x.size
    h = x[1] - x[0]
    integrand_ = integrand(x, A, k)
    integ = h/2 * (integrand_[0] + integrand_[1])
    
    for j in range(J-1):
        integ = integ + (h/2 * (integrand_[j] + integrand_[j+1]))

    return 2*integ

In [ ]:
w_st = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_st.set_init_condition(w_dark_st)
w_st.solve(b_tol=1e-4)#method="fixed point")

In [ ]:
np.random.seed(67)

def sol_gauss(x, A):
    output = w_dark_st(x) + A*np.exp(-x**2)
    return output

eps_rand = 1e-1
A_rand = np.random.rand(len(x_dark_st)) * eps_rand
A_rand[np.abs(x_dark_st)>=L/2] = eps_rand
#A = #2e-2 #1e-2#0.02041595129863531 #7e-2#roots[1]-0.02
#sigma=10
init_cond = lambda x: sol_gauss(x, A_rand)

In [ ]:
plt.plot(x_dark_st, norm(init_cond(x_dark_st)), "-", marker="none")
plt.xlim(-10, 10)

In [ ]:
plt.plot(x_dark_st, norm(sol_gauss(x_dark_st, A_rand)), ls="-", marker="none", label="perturbed soliton")
plt.plot(x_dark_st, norm(w_dark_st(x_dark_st)), ls="-", marker="none", label="Soliton")
plt.xlim(-10, 10)
plt.legend()
plt.show()

In [ ]:
w_st_gauss_rand = Def.CNFD(L, N, Nt, tf, pars, "neumann", order=2)
w_st_gauss_rand.set_init_condition(init_cond)
w_st_gauss_rand.solve(b_tol=1e-4)#method="fixed point")

In [ ]:
def theo_dark_st(x, t):
    output = w_dark_st(x - c_st*t)
    return output

In [ ]:
j = 50 #10000-1
plt.plot(w_st.x, norm(w_st_gauss_rand.w[j]), marker="none", ls="-", label="random") #norm(w_st.w[j])
#plt.plot(w_st.x, norm(theo_dark_st(w_st.x, w_st.t[j])), marker="none", ls="-", label="theo")
plt.legend()
plt.xlim(-10, 10)

plt.show()

In [ ]:
#w_st.w_theo = theo_dark_st
#w_st_gauss_007.w_theo = theo_dark_st
w_st_gauss_rand.w_theo = theo_dark_st
#w_in_gauss_001.w_theo = theo_dark_in
#w_in_gauss_0001.w_theo = theo_dark_in
#w_in_gauss_00001.w_theo = theo_dark_in

#L2_errors_st = w_st.get_error()#np.zeros(len(w_in.w)-1)
#L2_errors_st_gauss_007 = w_st_gauss_007.get_error()#np.zeros(len(w_in.w)-1)
L2_errors_st_gauss_rand = w_st_gauss_rand.get_error()
#errors_gauss_001 = w_in_gauss_001.get_error()#np.zeros(len(w_in.w)-1)
#errors_gauss_0001 = w_in_gauss_0001.get_error()#np.zeros(len(w_in.w)-1)
#errors_gauss_00001 = w_in_gauss_00001.get_error()#np.zeros(len(w_in.w)-1)


In [ ]:
#slope = m_t_0(x_dark_st, A, k)
#m_0 = norm(A_rand * (np.pi/2)**(1/4))

ms = .5

cond = norm(w_st_gauss_rand.w) != 0

norm_g = np.sqrt(w_st_gauss_rand.h) * np.linalg.norm(g(x_dark_st, A_rand), ord=2)

#plt.plot(0, m_0**2, ms=2)
#plt.axline((0, m_0**2), slope=slope, linewidth=1)
#plt.plot(w_st.t, L2_errors_st**2, ".", label="Standard", ms=ms)
#plt.plot(w_st.t, L2_errors_st_gauss_007, ".", label=r"$A={}$".format(A), ms=ms) #**2
plt.plot(w_st_gauss_rand.t[cond[:, 0]], L2_errors_st_gauss_rand[cond[:, 0]]/norm_g, ".-", label=r"$A=$random", ms=ms)
#plt.plot(w_in.t, errors_gauss_001, ".", label=r"Gauss $A=10^{-2}$", ms=ms)
#plt.plot(w_in.t, errors_gauss_0001, ".", label=r"Gauss $A=10^{-3}$", ms=ms)
#plt.plot(w_in.t, errors_gauss_00001, ".", label=r"Gauss $A=10^{-4}$", ms=ms)
plt.xlabel(r"$t$")
plt.ylabel(r"$L^2$-diff", rotation='horizontal', ha='right')
plt.title(r"Stable Dark soliton in $D_2$, $c\approx{c}, \kappa={k}, L={L}$".format(c=round(c_st, 2), k=k, L=L))
plt.xscale("log")
plt.yscale("log")
plt.legend()
#plt.xlim(-0.01, 0.55)
#plt.ylim(0, 0.025)
#plt.ylim(-0.01, 0.02)
#plt.xlim(-0.1, 0.2)
#plt.xlim(-0.01, 0.4)
#plt.xlim(-0.001, 0.02)
#plt.ylim(-0.001, 0.01)
#plt.savefig("diffs/Dark_D2.png", dpi=300, bbox_inches = "tight")
plt.show()

In [ ]:
plt.plot(x_dark_st, g(x_dark_st, A_rand), marker="none", ls="-")
plt.xlim(-3, 3)